# Gemini ve LangChain ile LLM API'larını Çağırma Giriş 🦜🔗

Bu notebook'ta LangChain aracılığıyla LLM API'larını nasıl kullanacağınızı öğreneceksiniz. Örnek olarak Google'ın Gemini API'sını kullanacağız. Bu notebook'un sonunda, LangChain kullanarak API çağrıları yapmayı ve bunu neden yaptığımızı bileceksiniz.

## ⚙️ Kurulum

👉 Kurulum aşamasında oluşturduğumuz `.env` dosyasındaki ortam değişkenlerini yüklemek için aşağıdaki hücreyi çalıştırın:

In [1]:
from dotenv import load_dotenv

load_dotenv() # Load environment variables from .env file

True

👉 Hücrenin çıktısı "`True`" mu? Harika! Artık Gemini API ile kimlik doğrulaması yapmak için kullanılacak bir `GOOGLE_API_KEY` ortam değişkeni kurmuş olduk.

Eğer değilse, yardım isteyin.

## Basit Bir API Çağrısı Yapma

Bu notebook'ta şunların nasıl yapılacağını göstereceğiz:
1. Google'ın kendi kütüphanesini kullanarak API çağrısı yapma.
2. Aynı işlemi LangChain kullanarak yapma.

## Google Generative AI Kütüphanesini Kullanma

In [2]:
from google import genai

In [3]:
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="What is the capital of France?",
)

`response` nesnesine bir göz atalım.

In [4]:
response.candidates[0].content.parts[0].text

'The capital of France is **Paris**.'

Gerçek cevabı nasıl alabileceğinizi görüyor musunuz?

Neyse ki, cevabı hemen almak için sadece `.text` özelliğini kullanabiliriz. Deneyin.

In [5]:
response.text

'The capital of France is **Paris**.'

Gemini cevaplarını Markdown formatında döndürür. Bunu kullanalım!

In [6]:
from IPython.display import Markdown
Markdown(response.text)

The capital of France is **Paris**.

Oluşturma parametrelerini de değiştirebilirsiniz. `google.genai` kullanarak bunu şu şekilde yaparsınız:

In [7]:
from google import genai
from google.genai import types # We need to import types for the config

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Write a social media post about how much you're learning about transformers.",
    config=types.GenerateContentConfig(
        max_output_tokens=200,
        temperature=1.0
    )
)

In [8]:
Markdown(response.text)

Here are a few options for a social media post about learning about transformers, choose the one that best fits your style and platform!

---

**Option 1: Enthusiastic & A Little Overwhelmed (Great for Twitter/Threads/Short Posts)**

> My brain is officially swimming in the wonderful, complex world of **Transformers**! 🤯 Seriously, the more I learn about these architectures, the more mind-blown I am. The attention mechanism is a game-changer. Soaking up all the knowledge! #AI #MachineLearning #Transformers #DeepLearning #NLP #Tech

---

**Option 2: Slightly More Technical & Focused (Good for LinkedIn/More Detailed Posts)**

> Deep diving into **Transformer** architectures this week and the insights are truly remarkable. Understanding the self-attention mechanism and how it enables contextual understanding across sequences has been a significant learning curve, but incredibly rewarding. Exciting to see the implications for NLP and beyond. #ArtificialIntelligence #

Harika. Ancak başka bir API denemek istediğinizi düşünün, örneğin OpenAI'nin veya Anthropic'in?

Onların dokümantasyonlarını incelemek ve tüm kodunuzu onların API'sini kullanacak şekilde yeniden yazmak zorunda kalırsınız. Tabii ki benzer olacaktır, ancak aynı olmayacaktır.

Neyse ki LangChain var!

## LangChain Kullanma 🦜🔗

Neden LangChain kullanırsınız?

1. **Model-Bağımsız Kod**

   LangChain, farklı LLM sağlayıcıları (Google, OpenAI, Anthropic, vb.) arasında minimal kod değişikliği ile geçiş yapmanızı sağlayan soyutlamalar sunar. Google API'sine doğrudan kod yazarsanız, sağlayıcı değiştirmek önemli ölçüde yeniden düzenleme gerektirir.

2. **Birleşik Arayüz**

   LangChain, altta yatan API'den bağımsız olarak farklı LLM sağlayıcıları arasında etkileşimleri standartlaştırır ve tutarlı yöntemler ile yanıt formatları sunar.

3. **Bileşenlerle Çalışabilirlik**

   LangChain'in zincir ve pipeline mimarisi, tüm alt yapıyı kendiniz halletmeden prompt, bellek ve erişim sistemlerini birleştiren karmaşık iş akışları oluşturmayı kolaylaştırır.

4. **Yerleşik Araçlar**

   LangChain, çıktı ayrıştırma, prompt şablonları ve kendiniz uygulamanız gereken diğer yardımcı araçları içerir.

[LangChain'in chat entegrasyonları listesi](https://docs.langchain.com/oss/python/integrations/chat)'ne gidin ve entegrasyon listesine bakın. Favori LLM sağlayıcınızı bulabiliyor musunuz?

Kodumuzda `chat_models.ChatGoogleGenerativeAI` kullanmak istemiyoruz çünkü bu özellikle Gemini için yapılmış. LLM'yi değiştirmek istersek, modeli başlatma şeklimizi değiştirmek zorunda kalırız. Neyse ki LangChain bir modeli başlatmak için daha genel bir yol sunar.

Gemini'yi tekrar kullanalım, ancak şimdi LangChain'in genel Chat Models'ini kullanarak.

👉 [LangChain'in "Models" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/models) sayfasına gidin ve Gemini kullanarak bir chat modelinin nasıl başlatılacağını bulun.

İpuçları:
1. Hemen "Basic Usage" bölümüne gidin.
2. Kullanmak istediğiniz modeli seçerek doğru dokümantasyonu hemen görebilirsiniz.

In [9]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

Modelin en temel kullanımı sadece `.invoke()` metodunu kullanmaktır:

In [11]:
response = model.invoke("köpeklerde idrar yolu enfeksiyonunu sebepleri nelerdir")

In [12]:
response.content

"Köpeklerde idrar yolu enfeksiyonlarının (İYE) birçok farklı sebebi olabilir. En yaygın nedenler şunlardır:\n\n**1. Bakteriyel Enfeksiyonlar (En Yaygın Sebep):**\n* **Escherichia coli (E. coli):** Köpeklerde en sık görülen idrar yolu enfeksiyonu etkenidir. Genellikle köpeğin anüs çevresindeki bakterilerin üretra yoluyla mesaneye ulaşmasıyla oluşur.\n* **Staphylococcus, Streptococcus, Proteus gibi diğer bakteriler:** Bu bakteriler de İYE'ye neden olabilir, ancak E. coli kadar sık görülmezler.\n\n**2. Anormal Anatomi veya Fizyoloji:**\n* **Doğumsal Kusurlar:** Bazı köpeklerde mesane veya üretra yapısında doğuştan gelen bozukluklar olabilir. Bu durumlar idrarın tam boşalmasını engelleyebilir ve bakterilerin üremesi için uygun ortam yaratabilir.\n* **Üretra Darlığı:** Üretra darlığı, idrar akışını zorlaştırarak enfeksiyon riskini artırabilir.\n* **Mesane Taşları (Urolithiasis):** Mesane taşları, idrar yolunda tahrişe ve bakterilerin yerleşmesi için pürüzlü bir yüzey oluşturabilir.\n* **Ano

Yanıta bir göz atalım. Nesnenin tüm öznitelik ve metodlarını içeren `__dict__`'ini güzel şekilde yazdırmak için `pprint()` kullanıyoruz.

In [13]:
from pprint import pprint
pprint(response.__dict__)

{'additional_kwargs': {},
 'content': 'Köpeklerde idrar yolu enfeksiyonlarının (İYE) birçok farklı '
            'sebebi olabilir. En yaygın nedenler şunlardır:\n'
            '\n'
            '**1. Bakteriyel Enfeksiyonlar (En Yaygın Sebep):**\n'
            '* **Escherichia coli (E. coli):** Köpeklerde en sık görülen idrar '
            'yolu enfeksiyonu etkenidir. Genellikle köpeğin anüs çevresindeki '
            'bakterilerin üretra yoluyla mesaneye ulaşmasıyla oluşur.\n'
            '* **Staphylococcus, Streptococcus, Proteus gibi diğer '
            "bakteriler:** Bu bakteriler de İYE'ye neden olabilir, ancak E. "
            'coli kadar sık görülmezler.\n'
            '\n'
            '**2. Anormal Anatomi veya Fizyoloji:**\n'
            '* **Doğumsal Kusurlar:** Bazı köpeklerde mesane veya üretra '
            'yapısında doğuştan gelen bozukluklar olabilir. Bu durumlar '
            'idrarın tam boşalmasını engelleyebilir ve bakterilerin üremesi '
            'için uygun orta

Cevabı çıkarın ve görüntüleyin. Markdown formatında olduğunu unutmayın, bu yüzden güzel görünmesini sağlayabilirsiniz.

In [14]:
from IPython.display import Markdown

Markdown(response.content)

Köpeklerde idrar yolu enfeksiyonlarının (İYE) birçok farklı sebebi olabilir. En yaygın nedenler şunlardır:

**1. Bakteriyel Enfeksiyonlar (En Yaygın Sebep):**
* **Escherichia coli (E. coli):** Köpeklerde en sık görülen idrar yolu enfeksiyonu etkenidir. Genellikle köpeğin anüs çevresindeki bakterilerin üretra yoluyla mesaneye ulaşmasıyla oluşur.
* **Staphylococcus, Streptococcus, Proteus gibi diğer bakteriler:** Bu bakteriler de İYE'ye neden olabilir, ancak E. coli kadar sık görülmezler.

**2. Anormal Anatomi veya Fizyoloji:**
* **Doğumsal Kusurlar:** Bazı köpeklerde mesane veya üretra yapısında doğuştan gelen bozukluklar olabilir. Bu durumlar idrarın tam boşalmasını engelleyebilir ve bakterilerin üremesi için uygun ortam yaratabilir.
* **Üretra Darlığı:** Üretra darlığı, idrar akışını zorlaştırarak enfeksiyon riskini artırabilir.
* **Mesane Taşları (Urolithiasis):** Mesane taşları, idrar yolunda tahrişe ve bakterilerin yerleşmesi için pürüzlü bir yüzey oluşturabilir.
* **Anormal Vajina Yapısı (Özellikle Dişi Köpeklerde):** Bazı dişi köpeklerde vajinanın normalden daha derin olması veya idrarın vajinaya kaçması (vaginal reflux) enfeksiyon riskini artırabilir.

**3. Hormonal Sorunlar:**
* **Yaşlanma ve Menopoz Sonrası Değişiklikler (Dişi Köpeklerde):** Östrojen seviyelerindeki düşüş, vajina ve üretra dokularının incelmesine ve daha hassas hale gelmesine neden olabilir, bu da enfeksiyonlara daha yatkınlık yaratır.
* **Hipotiroidizm:** Tiroid hormonlarının düşük olması bağışıklık sistemini zayıflatarak enfeksiyon riskini artırabilir.
* **Cushing Hastalığı (Hiperadrenokortisizm):** Aşırı kortizol üretimi bağışıklık sistemini baskılar ve İYE'ye yatkınlığı artırır.

**4. Bağışıklık Sistemi Zayıflığı:**
* **Kronik Hastalıklar:** Diyabet, böbrek hastalıkları gibi kronik rahatsızlıklar bağışıklık sistemini zayıflatabilir.
* **Kanser Tedavisi:** Kemoterapi gibi tedaviler bağışıklık sistemini baskılayabilir.
* **Yaşlılık:** Yaşlı köpeklerin bağışıklık sistemleri genellikle daha zayıftır.
* **Stres:** Uzun süreli stres bağışıklık fonksiyonlarını olumsuz etkileyebilir.

**5. İdrarın Tam Boşaltılamaması (İdrar Retansiyonu):**
* **Sinir Hasarı:** Omurilik yaralanmaları, disk hastalıkları veya diğer nörolojik sorunlar mesanenin sinir kontrolünü bozarak tam boşalmasını engelleyebilir.
* **Anestezi Sonrası:** Ameliyat sonrası anestezi etkisi geçene kadar mesane tam olarak boşalmayabilir.
* **Dış Bası:** Karın içindeki kitleler veya büyümüş organlar mesaneye bası yapabilir.

**6. İdrar Akışını Engelleyen Diğer Durumlar:**
* **Tümörler:** Mesane veya üretra tümörleri idrar akışını engelleyebilir.
* **İnflamasyon (Enfeksiyon Olmayan):** Bazı durumlarda enfeksiyon olmasa da mesane veya üretra iltihaplanabilir.

**7. İlaçlar:**
* **Steroidler:** Kortikosteroidler gibi bağışıklık sistemini baskılayan ilaçlar, enfeksiyonlara karşı savunmayı zayıflatabilir.

**8. Yaşam Tarzı Faktörleri:**
* **Yetersiz Tuvalet Eğitimi veya Molaları:** İdrarı uzun süre tutmak, bakterilerin üremesi için zaman kazandırabilir.
* **Sıcak ve Nemli Ortamlar:** Bazı bakteriler bu ortamlarda daha kolay ürerler.

**Özetle, köpeklerde idrar yolu enfeksiyonlarının temelinde genellikle bakterilerin üreme ortamı bulması yatar. Bu ortam ise genellikle idrarın tam boşaltılamaması, anatomik veya fizyolojik sorunlar, hormonal dengesizlikler veya zayıflamış bir bağışıklık sistemi tarafından sağlanır.**

Eğer köpeğinizde idrar yolu enfeksiyonu belirtileri (sık idrara çıkma, idrar yaparken zorlanma, idrarda kan, idrar kokusu değişikliği, iştahsızlık, halsizlik gibi) görüyorsanız, en kısa sürede bir veteriner hekime başvurmanız önemlidir. Doğru teşhis ve tedavi, köpeğinizin sağlığı için kritik öneme sahiptir.

Modelin temperature değerini `.temperature` özniteliğine erişerek kontrol edebilirsiniz. Deneyin:

In [15]:
model.temperature

0.7

Modeli kullanmadan önce, özniteliklere yeni değerler atayarak oluşturma parametrelerini de ayarlayabiliriz.

Daha önce Google'ın kütüphanesini kullanarak sosyal medya gönderisi yazmak için yaptığımızın eşdeğerini kodlamaya çalışın.

> _Not_: Normal olarak modelin `max_output_tokens` değerini ayarlayabilmemiz gerekir (modeli başlatırken veya daha sonra özniteliği değiştirerek). _langchain_google_genai_'nin mevcut sürümü (4.1.1) bir [hataya](https://github.com/langchain-ai/langchain-google/issues/1454) sahip ve bu çalışmıyor. Geçici çözüm? `max_output_tokens`'ı `.invoke()` metodunun bir parametresi olarak ayarlayın.

In [ ]:
# Set the maximum number of output tokens to 200

# YOUR CODE HERE

# Set the temperature to 1.0

# YOUR CODE HERE

# Generate a response with the new settings

# YOUR CODE HERE

# Display the response

# YOUR CODE HERE


In [16]:
# Set the temperature to 1.0
model.temperature = 1.0

# Generate a response with the new settings
response = model.invoke(
    "Write a social media post about artificial intelligence.",
    max_output_tokens=200
)

# Display the response
Markdown(response.content)

Here are a few options for a social media post about Artificial Intelligence, catering to different tones and platforms. Choose the one that best fits your style!

---

**Option 1: Enthusiastic & Forward-Looking (Good for LinkedIn, Twitter, Facebook)**

🚀 The future is here, and it's powered by Artificial Intelligence! From revolutionizing healthcare to unlocking new creative possibilities, AI is rapidly transforming our world. What's the most exciting AI advancement you've seen or are looking forward to? Let's discuss! #AI #ArtificialIntelligence #Innovation #FutureTech #TechTrends

---

**Option 2: Thought-Provoking & Curious (Good for Twitter, Instagram Story, Facebook)**

🤔 Thinking about AI today. It's incredible how quickly it's evolving, offering solutions to complex problems and opening doors we never imagined. But with all this power, what are the ethical considerations we need to keep front and center? What are your thoughts?

Bunun avantajı? Bu LangChain Chat Model birçok başka API'yi destekleyebilir.

Başka bir modele geçmek için değiştirmeniz gereken tek şeyler:
1. Diğer model için bir API anahtarı alın ve kodunuzda tanımlayın.
2. Modeli başlatırken model ve sağlayıcıyı değiştirin.

### Çoklu Mesajlar

`.invoke()` fonksiyonunu sadece tek bir mesajla kullanmak biraz kısıtlayıcı.

Şu gibi birden fazla mesaj sağlayabilirsiniz:
- `SystemMessage` veya sistem mesajları: modelin nasıl davranacağını söylemek için
- `HumanMessage` veya Kullanıcı mesajları: kullanıcıdan gelen girdi
- `AIMessage` veya Asistan mesajları: modelden gelen yanıt

Bir sosyal medya yazarı yapalım.

Modele nasıl davranacağını açıklayan bir sistem mesajı göndereceğiz. Sonra kullanıcı mesajında, kendimizi sadece yazacağı konuyu vermekle sınırlayabiliriz.

Bunu nasıl yapacağınızı öğrenmek için [LangChain'in "Messages" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/messages)'na bakın.

Sistem mesajı için ilhama mı ihtiyacınız var? İşte başlamanız için temel bir talimat:

```python
"""Sen Üretken AI öğrencisi için gönderiler yazan yaratıcı bir sosyal medya yazarısın.
Gönderilerinde her zaman kelime oyunu ve harekete geçirici çağrı bulunur.
Gönderilerin maksimum 200 karakter uzunluğundadır.
Her zaman emoji kullanırsın.
"""
```

In [ ]:
# Import the necessary classes

# YOUR CODE HERE

# Create a list of messages

# YOUR CODE HERE

# Generate a response using the list of messages

# YOUR CODE HERE

# Display the response

# YOUR CODE HERE


In [17]:
from langchain.messages import SystemMessage, HumanMessage
from IPython.display import Markdown

messages = [
    SystemMessage(
        """Sen Üretken AI öğrencisi için gönderiler yazan yaratıcı bir sosyal medya yazarısın.
Gönderilerinde her zaman kelime oyunu ve harekete geçirici çağrı bulunur.
Gönderilerin maksimum 200 karakter uzunluğundadır.
Her zaman emoji kullanırsın."""
    ),
    HumanMessage("Yapay zeka hakkında bir sosyal medya gönderisi yaz.")
]

response = model.invoke(messages)

Markdown(response.content)

Yapay zeka dünyasına adım atın, geleceği inşa edin! 🚀 Öğrenmeye ve yenilik yapmaya hazır mısınız? ✨ #YapayZeka #Gelecek #Teknoloji #Öğren

🏁 Tebrikler! Artık LangChain kullanarak çoklu mesajlarla temel prompt yazma konusunda uzmanlaştınız.